# Sistema de predicción y clasificación de oportunidades comerciales

**Capítulo 2 | Práctica guiada: 105 minutos | Visual Studio Code + Python**

Un equipo comercial necesita responder dos preguntas sobre las mismas oportunidades:

1. **¿Qué valor económico se estima?** Regresión lineal para predecir el valor esperado de referencia, en millones de COP.
2. **¿Qué probabilidad tienen de convertirse en venta?** Regresión logística para estimar la conversión y asignar una clase.

El valor esperado es la **expectativa económica definida en el caso**; se distingue del monto nominal de la oportunidad y del ingreso efectivamente cobrado.

Ejecute las celdas en orden, experimente en las líneas **MODIFIQUE AQUÍ** y complete las interpretaciones. Para responder, haga doble clic en la celda de texto; use **Shift+Enter** para visualizar y **Ctrl+S** para guardar.


<a id="preparacion"></a>
## 0. Preparación del entorno

**Tiempo de comprobación: 5 minutos.**

Abra **Capitulo02/materiales** en Visual Studio Code y seleccione **Select Kernel > Python Environments**. Elija el Python del curso y acepte la instalación de `ipykernel` si se solicita. La [preparación del entorno](../preparacion_entorno.md) contiene las indicaciones de instalación.

### 0A. Instalar dependencias

Ejecute la siguiente celda una vez. `%pip` instala las bibliotecas en el entorno del kernel seleccionado y muestra el proceso debajo de la celda. Espere a que finalice; si solicita **Restart Kernel**, reinicie y continúe con **0B**.


In [ ]:
# Instala las dependencias en el Python del kernel y muestra el progreso.
%pip install -r requirements.txt --progress-bar on


### 0B. Importar herramientas y cargar el caso

En esta celda se preparan las herramientas del laboratorio:

- **pandas:** lectura y análisis de oportunidades.
- **NumPy:** operaciones numéricas y vectorización.
- **Matplotlib:** gráficos para interpretar los resultados.
- **scikit-learn:** entrenamiento, escalamiento y evaluación.

También se carga `oportunidades_comerciales.csv` y se verifica su estructura mediante `assert`. Confirme **900 registros y 11 columnas**: ocho características, dos objetivos y un identificador.


In [ ]:
# Utilidades para localizar archivos, verificar su integridad y registrar versiones.
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import os
import sys
# Cálculo numérico, tablas y visualizaciones.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
# Herramientas para separar datos, entrenar y evaluar ambos modelos.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, log_loss, roc_auc_score,
)

# Mostramos cada gráfico debajo de su celda dentro de Jupyter.
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')

# La ruta se resuelve desde materiales, Capitulo02 o la raíz del proyecto.
def localizar_materiales() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for relativa in [Path('.'), Path('materiales'), Path('Capitulo02/materiales')]:
            candidata = base / relativa
            if (candidata / 'oportunidades_comerciales.csv').is_file():
                return candidata.resolve()
    raise FileNotFoundError('Abra Capitulo02/materiales y compruebe el CSV.')

MATERIALES = localizar_materiales()
RUTA_DATOS = MATERIALES / 'oportunidades_comerciales.csv'
# El instructor puede redirigir salidas; el participante usa resultados/.
SALIDAS = Path(os.environ.get('LAB2_OUTPUT_DIR', str(MATERIALES / 'resultados')))
SALIDAS.mkdir(parents=True, exist_ok=True)
# Conservamos la huella del CSV para comprobar al final que la fuente sigue intacta.
HASH_DATOS = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
datos = pd.read_csv(RUTA_DATOS)

# Entradas disponibles al corte: los objetivos se mantienen en columnas separadas.
CARACTERISTICAS = [
    'tamano_cliente_empleados', 'interacciones_comerciales',
    'duracion_proceso_dias', 'servicios_solicitados',
    'valor_oportunidad_millones_cop', 'reuniones_decisor',
    'propuesta_formal', 'relacion_cliente_meses',
]
OBJETIVO_REG = 'valor_esperado_millones_cop'
OBJETIVO_CLF = 'convertida'
# Los assert detienen la celda si los datos incumplen las condiciones del caso.
assert set(CARACTERISTICAS + [OBJETIVO_REG, OBJETIVO_CLF, 'oportunidad_id']) == set(datos)
assert len(datos) == 900 and datos.oportunidad_id.is_unique
assert not datos.isna().any().any()
assert np.isfinite(datos[CARACTERISTICAS + [OBJETIVO_REG, OBJETIVO_CLF]]).all().all()
assert set(datos[OBJETIVO_CLF].unique()) == {0, 1}
assert datos[OBJETIVO_REG].between(0, datos.valor_oportunidad_millones_cop).all()

pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Kernel:', sys.executable)
print('Python:', sys.version.split()[0])
print('Versiones:', {p: version(p) for p in ['numpy', 'pandas', 'matplotlib', 'scikit-learn']})
print(f'Dataset cargado: {len(datos)} registros | {datos.shape[1]} columnas')
print('Archivo:', RUTA_DATOS.name)
print('Preparación verificada: esquema completo y valores válidos.')


<a id="reto-1"></a>
## 1. Explorar el caso y separar los datos

**Tiempo de trabajo: 8 minutos.**

### Contexto del caso de uso

Antes de priorizar oportunidades, el equipo debe reconocer qué información estaba disponible al realizar la estimación. Las características describen ese momento; `convertida` registra el desenlace posterior: **1 = venta concretada; 0 = no concretada**.

### Qué hace el código

1. Separa **540 filas de entrenamiento**, **180 de validación** y **180 de prueba**.
2. Mantiene las mismas particiones para ambos modelos.
3. Muestra ejemplos, estadísticas y la distribución de clases.

El entrenamiento permite aprender; la validación, elegir configuraciones; la prueba, evaluar la elección final. Observe si las clases tienen cantidades similares y qué rangos presentan las entradas.


In [ ]:
# La semilla reproduce la partición; stratify conserva la proporción de clases.
SEMILLA = 42
# Reservamos primero la prueba y luego dividimos el 80% restante.
idx_desarrollo, idx_prueba = train_test_split(
    datos.index, test_size=0.20, random_state=SEMILLA, stratify=datos[OBJETIVO_CLF],
)
idx_entrenamiento, idx_validacion = train_test_split(
    idx_desarrollo, test_size=0.25, random_state=SEMILLA,
    stratify=datos.loc[idx_desarrollo, OBJETIVO_CLF],
)
datos_train = datos.loc[idx_entrenamiento].copy()
datos_val = datos.loc[idx_validacion].copy()
datos_test = datos.loc[idx_prueba].copy()

# Compartir la partición hace comparables ambos modelos sobre las mismas filas.
particiones = {'entrenamiento': datos_train, 'validacion': datos_val, 'prueba': datos_test}
display(pd.DataFrame([
    {'particion': nombre, 'filas': len(tabla), 'convertida_1': int(tabla.convertida.sum())}
    for nombre, tabla in particiones.items()
]))
# Revisamos ejemplos, rangos numéricos y cantidad de oportunidades por clase.
display(datos_train.head(5))
display(datos_train[CARACTERISTICAS + [OBJETIVO_REG]].describe().T.round(2))
display(datos_train.convertida.value_counts().sort_index().rename('cantidad').to_frame())


### Leer el gráfico de dispersión

Cada punto representa una oportunidad de entrenamiento. El **eje X** muestra su monto nominal y el **eje Y**, su valor esperado de referencia, ambos en millones de COP. Círculos y triángulos distinguen los desenlaces.

Un punto `(200, 80)` representa 200 millones nominales y 80 millones de expectativa. Observe la tendencia y la dispersión: montos similares pueden tener expectativas y desenlaces distintos.


In [ ]:
# Cada marcador es una oportunidad; separamos clases para observar desenlaces.
plt.figure(figsize=(8, 4.5))
for clase, marca, etiqueta in [(0, 'o', 'No convertida'), (1, '^', 'Convertida')]:
    # Filtramos cada clase; alpha controla la transparencia de sus puntos.
    grupo = datos_train.loc[datos_train.convertida == clase]
    plt.scatter(grupo.valor_oportunidad_millones_cop, grupo[OBJETIVO_REG],
                marker=marca, alpha=0.55, s=25, label=etiqueta)
plt.title('Valor nominal y expectativa económica: entrenamiento')
plt.xlabel('Valor nominal (millones COP)')
plt.ylabel('Valor esperado de referencia (millones COP)')
plt.legend()
plt.tight_layout()
plt.savefig(SALIDAS / '01_dispersion_valores.png', dpi=150)
plt.show()


<a id="reflexion-1"></a>

### Interpretación del problema integrado

Complete con los resultados obtenidos:

**1. ¿Qué variable predice la regresión y cuál la clasificación?**

Respuesta: ____________________

**2. ¿Qué representa un punto del gráfico y por qué dos montos similares pueden tener valores esperados diferentes?**

Respuesta: ____________________

**3. ¿Qué conjunto usará para elegir parámetros y cuál reservará para la evaluación final?**

Respuesta: ____________________


<a id="reto-2"></a>
## 2. Estimar el valor esperado mediante regresión

**Tiempo de trabajo: 12 minutos.**

### Contexto del caso de uso

Planeación necesita una estimación económica preliminar. Reutilizaremos la regresión del Capítulo 1 para comparar dos representaciones: las ocho entradas originales y una versión que agrega intensidad de contacto, valor por servicio e interacción entre contactos y propuesta.

### Qué hace el código

`crear_caracteristicas` calcula las entradas. Cada `Pipeline` aplica **StandardScaler → LinearRegression**, aprendiendo las transformaciones solo con entrenamiento. Ambos candidatos se evalúan en las mismas filas de validación.

Observe **MAE y RMSE**, expresados en millones de COP, y **R²**, que compara el error con predecir la media del conjunto evaluado. La regla selecciona el candidato con menor MAE de validación.


In [ ]:
def crear_caracteristicas(tabla: pd.DataFrame, ampliadas: bool = False) -> pd.DataFrame:
    """Aplica las mismas transformaciones a entrenamiento y oportunidades nuevas."""
    x = tabla[CARACTERISTICAS].copy()
    if ampliadas:
        # El +1 evita dividir por cero y estabiliza casos de pocos días.
        x['intensidad_comercial'] = x.interacciones_comerciales / (x.duracion_proceso_dias / 7 + 1)
        x['valor_por_servicio'] = x.valor_oportunidad_millones_cop / x.servicios_solicitados
        x['interaccion_propuesta'] = x.interacciones_comerciales * x.propuesta_formal
    return x

# MAE y RMSE conservan la unidad monetaria; R2 es adimensional.
def metricas_regresion(y_real, y_estimado) -> dict:
    return {'MAE': mean_absolute_error(y_real, y_estimado),
            'RMSE': np.sqrt(mean_squared_error(y_real, y_estimado)),
            'R2': r2_score(y_real, y_estimado)}

modelos_regresion, filas_regresion = {}, []
# Comparamos las entradas originales y ampliadas usando las mismas particiones.
for nombre, ampliar in [('base', False), ('ampliada', True)]:
    modelo = Pipeline([('escalador', StandardScaler()), ('regresion', LinearRegression())])
    # fit aprende el escalamiento y la regresión exclusivamente con entrenamiento.
    modelo.fit(crear_caracteristicas(datos_train, ampliar), datos_train[OBJETIVO_REG])
    pred_val = modelo.predict(crear_caracteristicas(datos_val, ampliar))
    modelos_regresion[nombre] = modelo
    filas_regresion.append({'modelo': nombre, 'particion': 'validacion',
                            **metricas_regresion(datos_val[OBJETIVO_REG], pred_val)})

comparacion_regresion = pd.DataFrame(filas_regresion)
display(comparacion_regresion)
# Regla acordada: menor MAE en validación; el conjunto de prueba sigue reservado.
NOMBRE_REG_FINAL = str(comparacion_regresion.sort_values(['MAE', 'modelo']).iloc[0]['modelo'])
modelo_regresion_final = modelos_regresion[NOMBRE_REG_FINAL]
USAR_AMPLIADAS = NOMBRE_REG_FINAL == 'ampliada'
print('Regresión seleccionada por MAE de validación:', NOMBRE_REG_FINAL)


### Del coeficiente al lenguaje comercial

La tabla ordena los coeficientes de la **regresión base** por magnitud. Cada peso indica el cambio estimado en millones de COP al aumentar una entrada en una desviación estándar, manteniendo las demás fijas.

El signo muestra la dirección de la asociación. Interprete estos pesos dentro del modelo y con criterio comercial: describen asociaciones aprendidas, no efectos causales.


In [ ]:
# Comparamos pesos estandarizados del mismo modelo base y sus signos.
coeficientes_regresion = pd.DataFrame({
    'caracteristica': CARACTERISTICAS,
    'coeficiente_por_desviacion': modelos_regresion['base'].named_steps['regresion'].coef_,
})
# Ordenar por magnitud permite localizar pesos grandes conservando su signo.
coeficientes_regresion['magnitud'] = coeficientes_regresion.coeficiente_por_desviacion.abs()
display(coeficientes_regresion.sort_values('magnitud', ascending=False))


<a id="reflexion-2"></a>

### Interpretación de la regresión comercial

Complete con los resultados obtenidos:

**1. ¿Cuál fue el MAE y el R² de la regresión base en validación? Exprese el error en millones de COP.**

Respuesta: ____________________

**2. ¿Qué cambió al agregar las tres características y qué modelo seleccionó la regla de menor MAE?**

Respuesta: ____________________

**3. Interprete el signo y la unidad de un coeficiente estandarizado. ¿Qué revisión comercial requeriría antes de usar el monto estimado?**

Respuesta: ____________________


<a id="reto-3"></a>
## 3. Experimentar con sigmoide, costo y gradiente

**Tiempo de trabajo: 17 minutos.**

### Contexto del caso de uso

Ahora queremos estimar si una oportunidad se convertirá en venta. La regresión logística combina las entradas en una puntuación `z = X @ w + b` y la transforma en una probabilidad mediante la **sigmoide**.

### 3A. Leer la sigmoide y la pérdida

La primera gráfica relaciona **puntuación y probabilidad**: en `z = 0`, la probabilidad es `0.5`. Agregue `-1` y `1` a `VALORES_Z`.

La segunda relaciona **probabilidad y pérdida**. Para una venta concretada, asignar `p = 0.1` se penaliza más que asignar `p = 0.9`. Observe cómo se invierte el comportamiento cuando la venta no se concreta.


In [ ]:
def sigmoide(z):
    """Evalúa la sigmoide de forma estable, incluso con puntuaciones extremas."""
    z = np.asarray(z, dtype=float)
    # logaddexp evita desbordamientos de exp para valores grandes de |z|.
    return np.exp(-np.logaddexp(0, -z))

VALORES_Z = [-4, -2, 0, 2, 4]  # MODIFIQUE AQUÍ: pruebe también -1 y 1.
display(pd.DataFrame({'z': VALORES_Z, 'probabilidad_conversion': sigmoide(VALORES_Z)}))
z_grafico = np.linspace(-8, 8, 250)
plt.figure(figsize=(8, 4))
plt.plot(z_grafico, sigmoide(z_grafico))
plt.axhline(0.5, linestyle='--', label='p = 0.5')
plt.axvline(0, linestyle=':')
plt.xlabel('Puntuación lineal z'); plt.ylabel('Probabilidad de conversión')
plt.title('Sigmoide: de puntuación a probabilidad'); plt.legend(); plt.tight_layout()
plt.savefig(SALIDAS / '02_sigmoide.png', dpi=150); plt.show()

# Comparamos la pérdida para ventas concretadas (1) y no concretadas (0).
p_grafico = np.linspace(0.01, 0.99, 200)
plt.figure(figsize=(8, 4))
plt.plot(p_grafico, -np.log(p_grafico), label='Resultado real = 1')
plt.plot(p_grafico, -np.log(1 - p_grafico), label='Resultado real = 0')
plt.xlabel('Probabilidad asignada a conversión'); plt.ylabel('Pérdida logística')
plt.title('El costo de equivocarse con mucha seguridad'); plt.legend(); plt.tight_layout()
plt.savefig(SALIDAS / '03_perdida_logistica.png', dpi=150); plt.show()


### 3B. Cómo aprende el modelo logístico

El equipo necesita probabilidades coherentes con los desenlaces históricos. Las funciones preparadas permiten observar cómo se ajusta el modelo:

- `costo_logistico`: mide la pérdida media y una penalización L2 opcional.
- `gradiente_logistico`: calcula los cambios del costo respecto a pesos e intercepto.
- `descenso_logistico`: actualiza esos parámetros y guarda el costo de cada iteración.

Usaremos **interacciones comerciales y días al corte**, escalados con entrenamiento. Cada ejecución comienza con pesos en cero. La tasa `alpha` controla el tamaño del ajuste; `lambda_l2 = 0` permite estudiar primero el aprendizaje sin penalización.


In [ ]:
def costo_logistico(X, y, w, b, lambda_l2=0.0) -> float:
    # Combinación vectorizada: una puntuación por oportunidad.
    z = X @ w + b
    # Pérdida logística calculada de forma numéricamente estable.
    perdida_media = np.mean(np.logaddexp(0, z) - y * z)
    # L2 penaliza pesos grandes; lambda_l2 = 0 deja solo la pérdida media.
    penalizacion = lambda_l2 * np.dot(w, w) / (2 * len(y))
    return float(perdida_media + penalizacion)

def gradiente_logistico(X, y, w, b, lambda_l2=0.0):
    # Diferencia entre la probabilidad estimada y el desenlace observado.
    error = sigmoide(X @ w + b) - y
    # X.T agrupa el aporte de todas las oportunidades a cada peso.
    dw = X.T @ error / len(y) + lambda_l2 * w / len(y)
    db = error.mean()  # El intercepto se ajusta separado de los pesos.
    return dw, float(db)

def descenso_logistico(X, y, alpha=0.1, iteraciones=300, lambda_l2=0.0):
    """Devuelve pesos, intercepto e historial, incluyendo el costo de iteración 0."""
    if not np.isfinite(alpha) or alpha <= 0 or iteraciones < 1 or lambda_l2 < 0:
        raise ValueError('Use alpha positivo, iteraciones >= 1 y lambda_l2 >= 0.')
    # Pesos e intercepto iniciales en cero para comparar tasas en igualdad.
    w = np.zeros(X.shape[1]); b = 0.0
    historial = [costo_logistico(X, y, w, b, lambda_l2)]
    for _ in range(iteraciones):
        # Calculamos el gradiente y actualizamos ambos parámetros con alpha.
        dw, db = gradiente_logistico(X, y, w, b, lambda_l2)
        w -= alpha * dw
        b -= alpha * db
        costo = costo_logistico(X, y, w, b, lambda_l2)
        if not np.isfinite(costo):
            raise FloatingPointError('Revise alpha y la escala de las entradas.')
        historial.append(costo)
    return w, b, np.array(historial)

# Dos entradas escaladas para observar el mecanismo de aprendizaje.
CARACTERISTICAS_2D = ['interacciones_comerciales', 'duracion_proceso_dias']
escalador_gd = StandardScaler()
X_gd = escalador_gd.fit_transform(datos_train[CARACTERISTICAS_2D])
y_gd = datos_train.convertida.to_numpy(dtype=float)


### 3C. Comparar ejecuciones

Compare `alpha = 0.01`, `0.10` y `0.50`; luego agregue `0.30` a `ALPHAS` y repita la celda.

En la gráfica, **X = iteración** y **Y = costo logístico promedio**. Los datos, los pesos iniciales y las 300 actualizaciones se mantienen iguales. Observe qué tasa reduce el costo más rápido y si la curva desciende de manera estable.


In [ ]:
ALPHAS = [0.01, 0.10, 0.50]  # MODIFIQUE AQUÍ: agregue 0.30.
ITERACIONES_GD = 300
historias_gd, filas_gd = {}, []
plt.figure(figsize=(8, 4.5))
# Cada llamada reinicia los pesos; solo cambia la tasa de aprendizaje.
for alpha in ALPHAS:
    w_gd, b_gd, historial = descenso_logistico(X_gd, y_gd, alpha, ITERACIONES_GD)
    historias_gd[alpha] = historial
    filas_gd.append({'alpha': alpha, 'costo_inicial': historial[0],
                    'costo_final': historial[-1],
                    'descenso_estable': bool(np.all(np.diff(historial) <= 1e-10))})
    plt.plot(range(len(historial)), historial, label=f'alpha = {alpha}')
resumen_gd = pd.DataFrame(filas_gd)
display(resumen_gd)
plt.xlabel('Iteración'); plt.ylabel('Costo logístico promedio')
plt.title('Aprendizaje del modelo de dos características')
plt.legend(); plt.tight_layout()
plt.savefig(SALIDAS / '04_descenso_gradiente.png', dpi=150); plt.show()


<a id="reflexion-3"></a>

### Interpretación de sigmoide, costo y gradiente

Complete con los resultados obtenidos:

**1. ¿Qué probabilidades obtuvo para z = -4, 0 y 4? ¿Qué representa z?**

Respuesta: ____________________

**2. Si una venta se concretó, ¿qué penaliza más la pérdida: p = 0.1 o p = 0.9? Explique por qué.**

Respuesta: ____________________

**3. ¿Qué observó al agregar alpha = 0.30? Compare costo final y estabilidad con las otras ejecuciones.**

Respuesta: ____________________


<a id="reto-4"></a>
## 4. Entrenar y evaluar la regresión logística

**Tiempo de trabajo: 17 minutos.**

### Contexto del caso de uso

El equipo comercial necesita orientar el seguimiento. Entrenaremos un clasificador con las ocho entradas y distinguiremos **probabilidad** de **clase**: una oportunidad se marca probable cuando su probabilidad alcanza el umbral.

### 4A. Calcular probabilidades y métricas

El `Pipeline` escala las entradas y ajusta `LogisticRegression`. `predict_proba` entrega probabilidades; `clasificar` aplica el umbral. Observe:

- **Exactitud:** proporción de clases correctas.
- **Precisión:** conversiones entre las marcadas probables.
- **Recall:** conversiones reales detectadas.
- **F1:** equilibrio entre precisión y recall.
- **ROC-AUC:** capacidad de ordenar oportunidades por conversión.
- **Log loss:** error de las probabilidades; menor es mejor.

Compare la exactitud con asignar a todas las oportunidades la clase mayoritaria de entrenamiento.


In [ ]:
# Obtenemos la probabilidad correspondiente a la clase de conversión.
def probabilidad_positiva(modelo, x) -> np.ndarray:
    # Encontramos explícitamente la columna de convertida=1.
    posicion = list(modelo.classes_).index(1)
    return modelo.predict_proba(x)[:, posicion]

def clasificar(probabilidades, umbral: float) -> np.ndarray:
    if not np.isfinite(umbral) or not 0 < umbral < 1:
        raise ValueError('El umbral debe estar entre 0 y 1, excluidos los extremos.')
    # Una probabilidad que alcanza el umbral recibe la clase 1.
    return (np.asarray(probabilidades) >= umbral).astype(int)

def metricas_clasificacion(y_real, probabilidades, umbral=0.5) -> dict:
    pred = clasificar(probabilidades, umbral)
    # La matriz cuenta aciertos y errores con filas reales y columnas predichas.
    vn, fp, fn, vp = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
    return {
        'umbral': float(umbral), 'VN': int(vn), 'FP': int(fp),
        'FN': int(fn), 'VP': int(vp), 'positivos_predichos': int(pred.sum()),
        'accuracy': accuracy_score(y_real, pred),
        'precision': precision_score(y_real, pred, zero_division=0),
        'recall': recall_score(y_real, pred, zero_division=0),
        'f1': f1_score(y_real, pred, zero_division=0),
        # Estas dos métricas usan probabilidades, no las clases del umbral.
        'roc_auc': roc_auc_score(y_real, probabilidades),
        'log_loss': log_loss(y_real, probabilidades, labels=[0, 1]),
    }

# El pipeline conserva el escalamiento aprendido para las predicciones posteriores.
modelo_clasificacion_base = Pipeline([
    ('escalador', StandardScaler()),
    ('logistica', LogisticRegression(C=1.0, solver='lbfgs', max_iter=4000, tol=1e-7)),
])
modelo_clasificacion_base.fit(datos_train[CARACTERISTICAS], datos_train.convertida)
prob_val_base = probabilidad_positiva(modelo_clasificacion_base, datos_val[CARACTERISTICAS])
metricas_base_val = metricas_clasificacion(datos_val.convertida, prob_val_base)
display(pd.DataFrame([metricas_base_val]))

# Referencia simple: asignar a todos la clase mayoritaria de entrenamiento.
clase_mayoritaria = int(datos_train.convertida.mode().iloc[0])
exactitud_referencia = float((datos_val.convertida == clase_mayoritaria).mean())
print(f'Exactitud de referencia en validación: {exactitud_referencia:.3f}')
print('Orden de clases devuelto por el modelo:', modelo_clasificacion_base.classes_)


### 4B. Leer la matriz de confusión y mover el umbral

**Filas = resultado real. Columnas = clase predicha.**

- **VP:** venta concretada marcada probable.
- **FP:** venta no concretada marcada probable; seguimiento que podría no rendir.
- **FN:** venta concretada marcada no probable; oportunidad que podría perder atención.
- **VN:** venta no concretada marcada no probable.

Cambie `UMBRAL_EXPERIMENTO` entre `0.35`, `0.50` y `0.65`. Al bajarlo, más oportunidades reciben seguimiento; al subirlo, la selección es más exigente. Compare los errores, la precisión y el recall. Los pesos y las probabilidades permanecen iguales: cambia la regla de clasificación.


In [ ]:
UMBRALES = [0.35, 0.50, 0.65]
UMBRAL_EXPERIMENTO = 0.50  # MODIFIQUE AQUÍ: compare 0.35 y 0.65.
# Reutilizamos las mismas probabilidades para comparar reglas de clasificación.
comparacion_umbrales = pd.DataFrame([
    metricas_clasificacion(datos_val.convertida, prob_val_base, u) for u in UMBRALES
])
display(comparacion_umbrales[['umbral', 'VP', 'FP', 'FN', 'VN',
                            'precision', 'recall', 'f1', 'positivos_predichos']])

# Dibujamos los conteos del umbral elegido: aciertos en la diagonal.
pred_val_experimento = clasificar(prob_val_base, UMBRAL_EXPERIMENTO)
matriz = confusion_matrix(datos_val.convertida, pred_val_experimento, labels=[0, 1])
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ConfusionMatrixDisplay(matriz, display_labels=['No probable (0)', 'Probable (1)']).plot(
    ax=ax, colorbar=False, values_format='d',
)
ax.set_yticklabels(['No convertida (0)', 'Convertida (1)'])
ax.set_xlabel('Clase predicha'); ax.set_ylabel('Resultado real')
ax.set_title(f'Validación: umbral = {UMBRAL_EXPERIMENTO:.2f}')
fig.tight_layout(); fig.savefig(SALIDAS / '05_matriz_validacion.png', dpi=150)
plt.show()


<a id="reflexion-4"></a>

### Interpretación de probabilidades y errores

Complete con los resultados obtenidos:

**1. Para el umbral 0.50, registre VP, FP, FN y VN. ¿Qué implica un falso negativo para el seguimiento comercial?**

Respuesta: ____________________

**2. ¿Cómo cambiaron la cantidad de probables, la precisión y el recall entre 0.35 y 0.65?**

Respuesta: ____________________

**3. ¿Qué umbral consideraría si se busca detectar más ventas potenciales? Justifique el costo de seguimiento adicional.**

Respuesta: ____________________


<a id="reto-5"></a>
## 5. Interpretar la frontera de decisión

**Tiempo de trabajo: 10 minutos.**

### Contexto del caso de uso

Para visualizar cómo se asigna una clase, usaremos un modelo didáctico con **interacciones comerciales y días al corte**.

### Qué debes observar

El eje X muestra contactos y el eje Y, días. El fondo representa probabilidades; los símbolos, desenlaces reales de validación. La línea marca dónde la probabilidad coincide con el umbral.

Cambie `UMBRAL_FRONTERA` de `0.50` a `0.65`: observe cómo se desplaza la frontera, conservando los pesos. Este dibujo representa dos entradas; el modelo comercial trabaja con ocho.


In [ ]:
# Modelo de dos entradas para visualizar la decisión en un plano.
modelo_frontera = Pipeline([
    ('escalador', StandardScaler()),
    ('logistica', LogisticRegression(C=1.0, max_iter=4000, tol=1e-7)),
])
modelo_frontera.fit(datos_train[CARACTERISTICAS_2D], datos_train.convertida)
# Construimos la malla con rangos de entrenamiento.
xx, yy = np.meshgrid(
    np.linspace(datos_train.interacciones_comerciales.min(), datos_train.interacciones_comerciales.max(), 160),
    np.linspace(datos_train.duracion_proceso_dias.min(), datos_train.duracion_proceso_dias.max(), 160),
)
malla = pd.DataFrame({CARACTERISTICAS_2D[0]: xx.ravel(), CARACTERISTICAS_2D[1]: yy.ravel()})
# Cada punto de la malla recibe una probabilidad para dibujar el fondo.
prob_malla = probabilidad_positiva(modelo_frontera, malla).reshape(xx.shape)
UMBRAL_FRONTERA = 0.50  # MODIFIQUE AQUÍ: compare con 0.65.
clasificar([0.5], UMBRAL_FRONTERA)  # Valida el parámetro.

plt.figure(figsize=(8, 5))
fondo = plt.contourf(xx, yy, prob_malla, levels=np.linspace(0, 1, 11), alpha=0.45)
plt.colorbar(fondo, label='Probabilidad estimada de conversión')
# La línea se traza donde la probabilidad alcanza el umbral seleccionado.
if prob_malla.min() < UMBRAL_FRONTERA < prob_malla.max():
    frontera = plt.contour(xx, yy, prob_malla, levels=[UMBRAL_FRONTERA], linewidths=2)
    plt.clabel(frontera, fmt=lambda v: f'p = {v:.2f}')
else:
    print('La frontera queda fuera del rango visualizado para este umbral.')
for clase, marca, etiqueta in [(0, 'o', 'Real 0'), (1, '^', 'Real 1')]:
    grupo = datos_val.loc[datos_val.convertida == clase]
    plt.scatter(grupo.interacciones_comerciales, grupo.duracion_proceso_dias,
                marker=marca, alpha=0.8, s=30, label=etiqueta)
plt.xlabel('Interacciones comerciales'); plt.ylabel('Días transcurridos al corte')
plt.title('Frontera del modelo didáctico de dos entradas')
plt.legend(); plt.tight_layout()
plt.savefig(SALIDAS / '06_frontera_decision.png', dpi=150); plt.show()


<a id="reflexion-5"></a>

### Interpretación de la frontera de decisión

Complete con los resultados obtenidos:

**1. ¿Qué representan el fondo, la línea y los marcadores del gráfico?**

Respuesta: ____________________

**2. ¿Qué cambió al mover el umbral a 0.65 y por qué una oportunidad cercana a la línea merece revisión?**

Respuesta: ____________________

**3. ¿Qué información comercial utiliza el clasificador principal que esta visualización de dos entradas deja fuera?**

Respuesta: ____________________


<a id="reto-6"></a>
## 6. Detectar sobreajuste y aplicar regularización

**Tiempo de trabajo: 18 minutos.**

### Contexto del caso de uso

Un modelo puede ajustarse muy bien a las oportunidades conocidas y fallar al evaluar otras. Para observarlo, usaremos **50 filas de entrenamiento**, dos entradas y una expansión polinómica de grado 8.

### Qué hace el código

Entrena la misma estructura con diferentes valores de `C` y compara entrenamiento con validación. En `LogisticRegression`, **C grande = regularización débil; C pequeño = regularización fuerte**. Su relación con la penalización es inversa.

Observe la **brecha de exactitud**, la pérdida y la magnitud de los pesos. Una brecha amplia sugiere sobreajuste; una penalización excesiva puede reducir el desempeño de ambos grupos. Buscamos generalización, no solo ajuste de entrenamiento.


In [ ]:
# La muestra reducida proviene solamente de entrenamiento.
idx_pequeno, _ = train_test_split(
    idx_entrenamiento, train_size=50, random_state=SEMILLA,
    stratify=datos_train.convertida,
)
X_pequeno = datos.loc[idx_pequeno, CARACTERISTICAS_2D]
y_pequeno = datos.loc[idx_pequeno, OBJETIVO_CLF]
# Muchos términos polinómicos y pocas filas permiten observar sobreajuste.
GRADO_POLINOMIO = 8
VALORES_C_DEMO = [1_000_000, 100, 1, 0.1, 0.01, 0.0001]
modelos_demo, filas_demo = {}, []
# C grande debilita L2; C pequeño refuerza la penalización de los pesos.
for c in VALORES_C_DEMO:
    modelo = Pipeline([
        ('escala_entrada', StandardScaler()),
        ('polinomio', PolynomialFeatures(degree=GRADO_POLINOMIO, include_bias=False)),
        ('escala_polinomio', StandardScaler()),
        ('logistica', LogisticRegression(C=c, max_iter=10000, tol=1e-7)),
    ])
    modelo.fit(X_pequeno, y_pequeno)
    # Comparamos ajuste sobre las 50 filas con generalización sobre validación.
    p_train = probabilidad_positiva(modelo, X_pequeno)
    p_val = probabilidad_positiva(modelo, datos_val[CARACTERISTICAS_2D])
    acc_train = accuracy_score(y_pequeno, clasificar(p_train, 0.5))
    acc_val = accuracy_score(datos_val.convertida, clasificar(p_val, 0.5))
    filas_demo.append({
        'C': c, 'accuracy_train': acc_train, 'accuracy_val': acc_val,
        'brecha': acc_train - acc_val,
        'log_loss_train': log_loss(y_pequeno, p_train),
        'log_loss_val': log_loss(datos_val.convertida, p_val),
        'norma_pesos': float(np.linalg.norm(modelo.named_steps['logistica'].coef_)),
        'iteraciones_solver': int(modelo.named_steps['logistica'].n_iter_[0]),
    })
    modelos_demo[c] = modelo
comparacion_regularizacion = pd.DataFrame(filas_demo)
display(comparacion_regularizacion)
print('Características transformadas:', modelo.named_steps['polinomio'].n_output_features_)


### Leer las curvas de regularización

El **eje X** muestra `C` en escala logarítmica; el **eje Y**, la exactitud. Hacia la derecha, la penalización se debilita.

Compare ambas curvas: una exactitud alta en entrenamiento y menor en validación indica una brecha de generalización. Busque el equilibrio considerando también el desempeño absoluto, no solo que las curvas estén próximas.


In [ ]:
# Ordenamos C para que el eje logaritmico aumente de izquierda a derecha.
orden_demo = comparacion_regularizacion.sort_values('C')
plt.figure(figsize=(8, 4.5))
# Las dos curvas permiten observar la brecha de generalización.
plt.semilogx(orden_demo.C, orden_demo.accuracy_train, marker='o', label='Entrenamiento: 50 filas')
plt.semilogx(orden_demo.C, orden_demo.accuracy_val, marker='s', label='Validación: 180 filas')
plt.xlabel('C: mayor valor, regularización más débil')
plt.ylabel('Exactitud'); plt.ylim(0, 1.05)
plt.title('Complejidad y generalización: experimento controlado')
plt.legend(); plt.tight_layout()
plt.savefig(SALIDAS / '07_regularizacion.png', dpi=150); plt.show()


### 6B. Seleccionar la configuración del modelo comercial

Retomamos el clasificador con **ocho entradas y 540 filas de entrenamiento**. La demostración polinómica anterior ayudó a estudiar el sobreajuste.

Esta celda compara valores de `C` y selecciona el de menor **log loss de validación**. Después muestra los efectos de varios umbrales sobre el modelo elegido.

Defina `UMBRAL_FINAL` antes del Reto 7 y justifique su elección con los falsos positivos y negativos observados. Regularización y umbral son decisiones diferentes.


In [ ]:
# Retomamos las ocho entradas y todas las filas de entrenamiento del caso comercial.
VALORES_C_COMERCIAL = [0.01, 0.1, 1.0, 10.0, 100.0]
modelos_comerciales, filas_comerciales = {}, []
for c in VALORES_C_COMERCIAL:
    modelo = Pipeline([
        ('escalador', StandardScaler()),
        ('logistica', LogisticRegression(C=c, max_iter=4000, tol=1e-7)),
    ])
    modelo.fit(datos_train[CARACTERISTICAS], datos_train.convertida)
    p_val = probabilidad_positiva(modelo, datos_val[CARACTERISTICAS])
    modelos_comerciales[c] = modelo
    filas_comerciales.append({'C': c, **metricas_clasificacion(datos_val.convertida, p_val)})
comparacion_c_comercial = pd.DataFrame(filas_comerciales)
display(comparacion_c_comercial[['C', 'log_loss', 'roc_auc', 'f1', 'accuracy']])

# Elegimos C por menor pérdida de validación, conservando la prueba para el cierre.
C_FINAL = float(comparacion_c_comercial.sort_values(['log_loss', 'C']).iloc[0]['C'])
modelo_clasificacion_final = modelos_comerciales[C_FINAL]
prob_val_final = probabilidad_positiva(modelo_clasificacion_final, datos_val[CARACTERISTICAS])
# Con el modelo elegido, revisamos cómo el umbral cambia los errores.
umbrales_modelo_final = pd.DataFrame([
    metricas_clasificacion(datos_val.convertida, prob_val_final, u) for u in UMBRALES
])
display(umbrales_modelo_final[['umbral', 'VP', 'FP', 'FN', 'VN', 'precision', 'recall', 'f1']])
UMBRAL_FINAL = 0.50  # MODIFIQUE AQUÍ antes del Reto 7: 0.35, 0.50 o 0.65.
clasificar(prob_val_final, UMBRAL_FINAL)
print(f'Configuración para evaluación final: C = {C_FINAL}, umbral = {UMBRAL_FINAL}')


<a id="reflexion-6"></a>

### Interpretación de sobreajuste y regularización

Complete con los resultados obtenidos:

**1. En el experimento de 50 filas, ¿qué C muestra mayor brecha y qué evidencia aporta log loss?**

Respuesta: ____________________

**2. ¿Qué ocurre con los pesos y el rendimiento al fortalecer mucho la regularización?**

Respuesta: ____________________

**3. ¿Qué C seleccionó el modelo comercial, con qué criterio, y qué umbral final eligió? Justifique la revisión de FP y FN.**

Respuesta: ____________________


<a id="reto-7"></a>
## 7. Evaluar e integrar predicciones de nuevas oportunidades

**Tiempo de trabajo: 13 minutos.**

### Contexto del caso de uso

Con los modelos y el umbral fijados, evaluaremos las **180 oportunidades de prueba**. Este resultado resume el comportamiento sobre filas reservadas durante la selección.

Compare las métricas finales y la tabla por oportunidad. Distinga el error monetario de los errores de clasificación antes de analizar los casos nuevos.


In [ ]:
# Evaluamos sobre prueba despues de fijar modelo, C y umbral con validacion.
pred_reg_test = modelo_regresion_final.predict(crear_caracteristicas(datos_test, USAR_AMPLIADAS))
prob_test = probabilidad_positiva(modelo_clasificacion_final, datos_test[CARACTERISTICAS])
pred_clf_test = clasificar(prob_test, UMBRAL_FINAL)
metricas_reg_test = metricas_regresion(datos_test[OBJETIVO_REG], pred_reg_test)
metricas_clf_test = metricas_clasificacion(datos_test.convertida, prob_test, UMBRAL_FINAL)
display(pd.DataFrame([{'modelo': NOMBRE_REG_FINAL, 'particion': 'prueba', **metricas_reg_test}]))
display(pd.DataFrame([{'C': C_FINAL, 'particion': 'prueba', **metricas_clf_test}]))

# Conservamos ambas estimaciones junto al resultado real de cada oportunidad.
predicciones_prueba = pd.DataFrame({
    'oportunidad_id': datos_test.oportunidad_id.to_numpy(),
    'valor_referencia_millones_cop': datos_test[OBJETIVO_REG].to_numpy(),
    'valor_estimado_millones_cop': pred_reg_test,
    'convertida_real': datos_test.convertida.to_numpy(),
    'probabilidad_conversion': prob_test,
    'clase_predicha': pred_clf_test,
})
display(predicciones_prueba.head(5))


### 7B. Obtener las dos predicciones de una oportunidad nueva

`predecir_oportunidades` valida las ocho entradas y devuelve **valor esperado en millones de COP, probabilidad, clase y observaciones de revisión**. También identifica entradas fuera de los rangos históricos y montos estimados que requieren revisión.

Compare los escenarios A, B y C. En C, cambie `interacciones_comerciales` de 4 a 8 y observe las salidas.

El monto ya representa una expectativa económica: interprételo junto a la probabilidad, sin volver a multiplicarlos. Una probabilidad puede variar y mantener la misma clase. Ambas estimaciones requieren valoración comercial.


In [ ]:
def predecir_oportunidades(nuevas: pd.DataFrame) -> pd.DataFrame:
    """Valida entradas y devuelve estimación, probabilidad, clase y revisiones."""
    nuevas = nuevas.reset_index(drop=True).copy()
    # Revisamos columnas, tipos y rangos antes de utilizar los modelos.
    faltantes = [c for c in CARACTERISTICAS if c not in nuevas]
    if faltantes:
        raise ValueError(f'Complete las características requeridas: {faltantes}')
    x = nuevas[CARACTERISTICAS].copy()
    for c in CARACTERISTICAS:
        x[c] = pd.to_numeric(x[c], errors='raise')
    if len(x) == 0 or not np.isfinite(x).all().all():
        raise ValueError('Ingrese al menos una oportunidad con valores numéricos finitos.')
    positivos = ['tamano_cliente_empleados', 'servicios_solicitados', 'valor_oportunidad_millones_cop']
    if (x[positivos] <= 0).any().any() or (x < 0).any().any():
        raise ValueError('Revise valores negativos o cero en empleados, servicios y monto.')
    enteros = [c for c in CARACTERISTICAS if c != 'valor_oportunidad_millones_cop']
    if not (x[enteros] == np.floor(x[enteros])).all().all():
        raise ValueError('Use cantidades enteras para conteos, días, meses y propuesta.')
    if not x.propuesta_formal.isin([0, 1]).all():
        raise ValueError('propuesta_formal debe ser 0 o 1.')
    # Cada modelo conserva las mismas transformaciones usadas al entrenar.
    resultado = nuevas.copy()
    resultado['valor_esperado_predicho_millones_cop'] = modelo_regresion_final.predict(
        crear_caracteristicas(x, USAR_AMPLIADAS))
    resultado['probabilidad_conversion'] = probabilidad_positiva(modelo_clasificacion_final, x)
    # La clase se obtiene al aplicar el umbral fijado en validación.
    resultado['clase_0_1'] = clasificar(resultado.probabilidad_conversion, UMBRAL_FINAL)
    resultado['clasificacion'] = np.where(resultado.clase_0_1 == 1, 'Probable', 'No probable')
    # Revisamos extrapolación variable a variable; combinaciones nuevas también requieren criterio.
    limites = datos_train[CARACTERISTICAS].agg(['min', 'max'])
    revisiones = []
    for i, fila in x.iterrows():
        fuera = [c for c in CARACTERISTICAS if not limites.loc['min', c] <= fila[c] <= limites.loc['max', c]]
        monto = resultado.loc[i, 'valor_esperado_predicho_millones_cop']
        motivos = [f'Entrada fuera de rango: {c}' for c in fuera]
        if not 0 <= monto <= fila.valor_oportunidad_millones_cop:
            motivos.append('Revisar estimación monetaria respecto del valor nominal')
        revisiones.append('; '.join(motivos) if motivos else 'Entradas dentro de rangos; validar contexto comercial')
    resultado['revision_humana'] = revisiones
    return resultado

# Tres escenarios para comparar valor económico, probabilidad y revisiones.
nuevas_oportunidades = pd.DataFrame([
    {'escenario': 'A - avance sólido', 'tamano_cliente_empleados': 2200,
     'interacciones_comerciales': 10, 'duracion_proceso_dias': 28,
     'servicios_solicitados': 3, 'valor_oportunidad_millones_cop': 280,
     'reuniones_decisor': 3, 'propuesta_formal': 1, 'relacion_cliente_meses': 18},
    {'escenario': 'B - maduración', 'tamano_cliente_empleados': 3500,
     'interacciones_comerciales': 6, 'duracion_proceso_dias': 62,
     'servicios_solicitados': 5, 'valor_oportunidad_millones_cop': 360,
     'reuniones_decisor': 1, 'propuesta_formal': 1, 'relacion_cliente_meses': 8},
    {'escenario': 'C - exploratoria', 'tamano_cliente_empleados': 480,
     'interacciones_comerciales': 4,  # MODIFIQUE AQUÍ: pruebe 8 interacciones.
     'duracion_proceso_dias': 50, 'servicios_solicitados': 2,
     'valor_oportunidad_millones_cop': 180, 'reuniones_decisor': 1,
     'propuesta_formal': 0, 'relacion_cliente_meses': 6},
])
predicciones_nuevas = predecir_oportunidades(nuevas_oportunidades)
display(predicciones_nuevas[['escenario', 'valor_esperado_predicho_millones_cop',
                            'probabilidad_conversion', 'clasificacion', 'revision_humana']])


### 7C. Comparar una sola variable y conservar la evidencia

La celda copia el escenario C y agrega **cuatro contactos** a su valor actual, conservando las demás entradas. Compare monto, probabilidad y clase: el cambio muestra sensibilidad del modelo, no una garantía del efecto de contactar al cliente.

Las tablas y la configuración se guardan en `resultados/`. Complete **Interpretación de las predicciones integradas** y guarde el notebook con **Ctrl+S**.


In [ ]:
# Conservamos las entradas de C y modificamos solo el número de contactos.
caso_antes = nuevas_oportunidades.iloc[[2]].copy()
caso_despues = caso_antes.copy()
caso_despues['interacciones_comerciales'] += 4
caso_despues['escenario'] = 'C - cuatro interacciones adicionales'
comparacion_escenario = predecir_oportunidades(pd.concat([caso_antes, caso_despues], ignore_index=True))
display(comparacion_escenario[['escenario', 'interacciones_comerciales',
                             'valor_esperado_predicho_millones_cop', 'probabilidad_conversion', 'clasificacion']])

# Cada archivo conserva su unidad y el conjunto de evaluación utilizado.
tablas_salida = {
    'regresion_validacion.csv': comparacion_regresion,
    'coeficientes_regresion_base.csv': coeficientes_regresion,
    'descenso_gradiente.csv': resumen_gd,
    'umbrales_clasificador_base.csv': comparacion_umbrales,
    'regularizacion_demo.csv': comparacion_regularizacion,
    'regularizacion_comercial.csv': comparacion_c_comercial,
    'umbrales_clasificador_final.csv': umbrales_modelo_final,
    'metricas_regresion_prueba.csv': pd.DataFrame([metricas_reg_test]),
    'metricas_clasificacion_prueba.csv': pd.DataFrame([metricas_clf_test]),
    'predicciones_prueba.csv': predicciones_prueba,
    'predicciones_nuevas_oportunidades.csv': predicciones_nuevas,
    'comparacion_escenario.csv': comparacion_escenario,
}
# Exportamos las tablas calculadas durante esta ejecución.
for nombre, tabla in tablas_salida.items():
    tabla.to_csv(SALIDAS / nombre, index=False, encoding='utf-8-sig')

# Registramos las decisiones y versiones necesarias para reproducir los resultados.
resumen_ejecucion = {
    'semilla_particion': SEMILLA, 'filas': len(datos),
    'particiones': {nombre: len(tabla) for nombre, tabla in particiones.items()},
    'regresion_elegida': NOMBRE_REG_FINAL, 'C_final': C_FINAL, 'umbral_final': UMBRAL_FINAL,
    'datos_sha256': HASH_DATOS, 'version_python': sys.version.split()[0],
    'versiones': {p: version(p) for p in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'ipykernel']},
}
(SALIDAS / 'configuracion_ejecucion.json').write_text(
    json.dumps(resumen_ejecucion, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Se guardaron {len(tablas_salida)} tablas y la configuración de ejecución.')


<a id="reflexion-7"></a>

### Interpretación de las predicciones integradas

Complete con los resultados obtenidos:

**1. Registre el MAE de regresión y el recall de clasificación sobre prueba. ¿Qué aspecto del resultado describe cada uno?**

Respuesta: ____________________

**2. Elija una oportunidad nueva y explique su valor esperado estimado, probabilidad y clase con el umbral utilizado.**

Respuesta: ____________________

**3. ¿Qué cambió al agregar cuatro interacciones al caso C? ¿Qué revisaría una persona antes de priorizarlo?**

Respuesta: ____________________


<a id="cierre"></a>
## 8. Cierre y verificación

**Tiempo: 5 minutos.**

La celda comprueba particiones, predicciones, probabilidades, archivo fuente y resultados guardados. El mensaje final confirma consistencia técnica.

Complete **Conclusiones de la práctica** relacionando la selección de modelos, los errores de clasificación y la supervisión humana. Guarde las salidas y sus respuestas como evidencia de aprendizaje.


In [ ]:
# Comprobamos particiones separadas y ausencia de objetivos entre las entradas.
assert len(set(idx_entrenamiento) & set(idx_validacion)) == 0
assert len(set(idx_entrenamiento) & set(idx_prueba)) == 0
assert len(set(idx_validacion) & set(idx_prueba)) == 0
assert len(idx_entrenamiento) + len(idx_validacion) + len(idx_prueba) == len(datos)
assert set(CARACTERISTICAS).isdisjoint({OBJETIVO_REG, OBJETIVO_CLF, 'oportunidad_id'})
# Las predicciones deben ser finitas y las probabilidades estar entre 0 y 1.
assert np.isfinite(pred_reg_test).all() and np.isfinite(prob_test).all()
assert np.all((prob_test >= 0) & (prob_test <= 1))
assert sum(metricas_clf_test[c] for c in ['VP', 'FP', 'FN', 'VN']) == len(datos_test)
assert len(predicciones_nuevas) == 3
# Verificamos que el CSV original se conserva y las tablas fueron exportadas.
assert hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest() == HASH_DATOS
assert all((SALIDAS / nombre).is_file() for nombre in tablas_salida)
print('Validación técnica completada: datos, modelos y archivos consistentes.')
print('Guarde también sus interpretaciones con Ctrl+S.')


<a id="reflexion-8"></a>

### Conclusiones de la práctica

Complete con los resultados obtenidos:

**1. ¿Qué reutilizó de regresión y qué incorporó al trabajar con clasificación?**

Respuesta: ____________________

**2. ¿Qué dos evidencias usaría para explicar al equipo por qué acertar en entrenamiento no basta?**

Respuesta: ____________________

**3. ¿Cómo combinaría la estimación económica y la probabilidad en una decisión de seguimiento con revisión humana?**

Respuesta: ____________________


## Apoyo opcional con Copilot Chat Standard

Comparta un bloque de código o sus resultados y contraste la explicación con el notebook.

```text
Estoy estudiando regresión y clasificación de oportunidades comerciales.
Este es el código o resultado que quiero comprender:
[PEGAR CÓDIGO O RESULTADOS]
Explica las operaciones importantes y su utilidad en el caso comercial.
Distingue valor esperado, probabilidad y clase cuando corresponda.
Usa solo los valores compartidos y termina con una pregunta de comprobación.
```

## Recursos técnicos

- [Curso de Coursera](https://www.coursera.org/learn/machine-learning).
- [Regresión logística](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.LogisticRegression.html).
- [Métricas de clasificación](https://scikit-learn.org/1.8/modules/model_evaluation.html#classification-metrics).
- [Separación de datos y fuga de información](https://scikit-learn.org/1.8/common_pitfalls.html).
